# Prediction from a polynomial

A prediction interval at $x_0$ is wider than a confidence interval for the mean of the polynomial, and both are wider outside the sample than at its center. The polynomial makes the variance formula a quadratic form in $(1, x_0, x_0^2)$.


## The two standard errors

Let $r(x_0) = (1,\ x_0,\ x_0^2)$. The estimated mean on the fitted polynomial is $r(x_0)\widehat\beta$, and

$$
\operatorname{se}\!\left(r(x_0)\widehat\beta\right)
= s\sqrt{r(x_0)\,(X^{\mathsf T}X)^{-1}\,r(x_0)^{\mathsf T}}.
$$

A new observation adds one error of variance $\sigma^2$, independent of the fit:

$$
\operatorname{se}(y_{\mathrm{new}})
= s\sqrt{1 + r(x_0)\,(X^{\mathsf T}X)^{-1}\,r(x_0)^{\mathsf T}}.
$$

Using the inverse from the inference lesson and $s^2 = 5$, the quadratic form at two places is short enough to finish by hand.

At $x_0 = 0$, the row is $(1, 0, 0)$, so the quadratic form is the $(1, 1)$ entry $17/35$. The fitted value is $b_0 = 1$, and

$$
\begin{aligned}
\operatorname{se}(\widehat\mu(0)) &= \sqrt{5\cdot\dfrac{17}{35}} = \sqrt{\dfrac{17}{7}}, \\[4pt]
\operatorname{se}(y_{\mathrm{new}}) &= \sqrt{5\left(1 + \dfrac{17}{35}\right)} = \sqrt{\dfrac{52}{7}}.
\end{aligned}
$$

At $x_0 = 3$, outside the sample $[-2, 2]$, the row is $(1, 3, 9)$ and the quadratic form equals $23/5$. The point prediction is

$$
1 + 2\cdot 3 + 3\cdot 9 = 34.
$$

The standard errors collapse cleanly:

$$
\begin{aligned}
\operatorname{se}(\widehat\mu(3)) &= \sqrt{5\cdot\dfrac{23}{5}} = \sqrt{23}, \\[4pt]
\operatorname{se}(y_{\mathrm{new}}) &= \sqrt{5\left(1 + \dfrac{23}{5}\right)} = \sqrt{28} = 2\sqrt{7}.
\end{aligned}
$$

The factor $23/5 = 4.6$ is larger than $1$. That is allowed. This factor is not a diagonal entry of the hat matrix, because $x = 3$ is not one of the five rows. Inside the sample the corresponding factors are at most $1$. At $x = -2$, for example, the factor is $31/35$.


In [1]:
# Variance factors and standard errors at x=0 and x=3.
import math
import numpy as np
gram_inv = np.array([
    [17 / 35, 0, -1 / 7],
    [0, 1 / 10, 0],
    [-1 / 7, 0, 1 / 14],
])

def factor(x0):
    row = np.array([1.0, x0, x0 ** 2])
    return float(row @ gram_inv @ row)

s = math.sqrt(5)
print("factor at 0 and 3", factor(0), factor(3))
print("se mean, se new at 0", s * math.sqrt(factor(0)), s * math.sqrt(1 + factor(0)))
print("se mean, se new at 3", s * math.sqrt(factor(3)), s * math.sqrt(1 + factor(3)))
assert abs(factor(0) - 17 / 35) < 1e-12
assert abs(factor(3) - 23 / 5) < 1e-12
assert abs(factor(-2) - 31 / 35) < 1e-12
assert abs(s * math.sqrt(factor(3)) - math.sqrt(23)) < 1e-12
assert abs(s * math.sqrt(1 + factor(3)) - 2 * math.sqrt(7)) < 1e-12
assert factor(3) > 1 > factor(-2)


factor at 0 and 3 0.4857142857142857 4.6000000000000005
se mean, se new at 0 1.5583874449479593 2.7255405754769875
se mean, se new at 3 4.79583152331272 5.291502622129181


## The intervals at the edge of the sample's story

With $t_{2,\,0.975}$ the same quantile as in the coefficient lesson, the $95\%$ intervals at $x = 3$ are

$$
34 \pm t_{2,\,0.975}\sqrt{23}
\qquad\text{and}\qquad
34 \pm t_{2,\,0.975}\cdot 2\sqrt{7}.
$$

The prediction interval is the wider one. Both are intervals around the quadratic $1 + 2x + 3x^2$, not around the cubic that happens to interpolate the five points. At $x = 3$ those two polynomials already disagree:

$$
1 + \dfrac{29}{6}\cdot 3 + 3\cdot 9 - \dfrac{5}{6}\cdot 27 = 20.
$$

The quadratic says $34$ and the cubic says $20$. No standard error computed inside one of those models includes the possibility that the other degree is the right one. Degree is a modeling choice made before the interval is drawn.


In [2]:
# At x=3 the quadratic predicts 34 and the cubic predicts 20.
from fractions import Fraction
import math
import scipy.stats as stats
quadratic = 1 + 2 * 3 + 3 * 9
cubic = 1 + Fraction(29, 6) * 3 + 3 * 9 - Fraction(5, 6) * 27
half = stats.t.ppf(0.975, 2) * 2 * math.sqrt(7)
print("quadratic", quadratic, "cubic", float(cubic))
print("prediction half-width", half)
assert quadratic == 34 and cubic == 20
assert half > 0


quadratic 34 cubic 20.0
prediction half-width 22.76749820158056


## Pitfall

The prediction $34$ is the quadratic evaluated at $x = 3$. The sample never contained an input larger than $2$. The standard error $\sqrt{23}$ for the mean, and $2\sqrt{7}$ for a new draw, describe uncertainty in the fitted quadratic under the assumption that the mean really is a quadratic all the way to $x = 3$. They become optimistic exactly when that assumption fails. The cubic's value $20$ is one concrete failure of carrying the quadratic formula past the last observation.


In [3]:
# x=3 is outside the observed inputs. The two polynomials differ by 14 there.
observed = [-2, -1, 0, 1, 2]
print("3 is outside", 3 not in observed)
print("absolute gap", abs(34 - 20))
assert 3 not in observed
assert abs(34 - 20) == 14


3 is outside True
absolute gap 14


## Summary

- At $x = 0$ the fitted mean is $1$, with standard error $\sqrt{17/7}$. A new observation has standard error $\sqrt{52/7}$.
- At $x = 3$ the quadratic predicts $34$, with standard errors $\sqrt{23}$ and $2\sqrt{7}$.
- The cubic through the same five points predicts $20$ at $x = 3$. An interval does not repair a degree chosen before the interval.
